# Qoro × PushQuantum Hackathon — 02 · Network design (QOBLIB 08)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QoroQuantum/pushquantum-hackathon-2026/blob/main/02_network_design.ipynb)

Build a directed network in which every node has exactly two outgoing and two incoming links, so that
when all traffic is routed, the busiest link carries as little as possible. A QUBO proposes network
topologies, and an exact linear program scores them. This notebook builds that pipeline, compares
quantum and classical proposers, and writes a solution file in QOBLIB's format.

**Your challenge: scale up.** Instances run from `network05` (20 binary variables) to `network24`
(552). Up to `network10` the optimum is proven; from `network11` on, only best-known values exist
(section 1.5). Solve the largest instance you can, as close to the best-known value as you can.

**Scoring** uses QOBLIB's official objective, not your QUBO's energy: `qt.route_min_congestion`
reproduces every published QOBLIB value. Treat it as the referee.

**Before you start,** work through notebook 00 (setup and QUBO 101).

---
## 0 · Setup

Run these cells first. Notebook 00 (Part 0) explains them: your API key and the install. The helper file
`qoblib_tools.py` is downloaded for you.

In [1]:
# Setup, explained in notebook 00 (Part 0). If this cell installs Divi, restart the session
# (Runtime → Restart session) and run the notebook from the top.
import os
from importlib.metadata import PackageNotFoundError, version

try:
    divi_version = version("qoro-divi")
except PackageNotFoundError:
    divi_version = None
if divi_version != "0.16.2":
    %pip install -q "qoro-divi[qubo-decompose,jupyter]==0.16.2"
    print("Installed Divi 0.16.2: restart the session and run from the top.")

try:  # Google Colab: copy the secret QORO_API_KEY into the environment
    from google.colab import userdata
    os.environ["QORO_API_KEY"] = userdata.get("QORO_API_KEY")
except ImportError:
    pass  # not in Colab: Divi reads .env or the environment by itself
except Exception as err:  # no secret yet, or notebook access switched off
    print(f"No QORO_API_KEY in Colab Secrets yet ({type(err).__name__}): see notebook 00, section 0.1.")

In [2]:
# The helper file qoblib_tools.py (instance loaders and official scorers) is downloaded from the
# hackathon repository if it is not next to this notebook yet.
import urllib.request

if not os.path.exists("qoblib_tools.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/QoroQuantum/pushquantum-hackathon-2026/main/qoblib_tools.py", "qoblib_tools.py")
import qoblib_tools as qt

missing = [name for name in [
    "load_network_demand", "route_min_congestion", "build_network_qubo", "repair_topology",
    "route_integral", "write_network_solution", "check_network_solution",
] if not hasattr(qt, name)]
if missing:
    raise ImportError(
        f"{os.path.relpath(qt.__file__)} is an old copy (no {', '.join(missing)}). Delete it, restart the "
        "session and run from the top: the current version is downloaded."
    )
print("qoblib_tools loaded from", os.path.relpath(qt.__file__))

qoblib_tools loaded from qoblib_tools.py


In [3]:
import itertools
import time
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)  # noisy docstrings in some dependencies

import hybrid
import numpy as np
import requests
from dwave.samplers import SimulatedAnnealingSampler
from IPython.display import display

from divi.backends import JobConfig, MaestroSimulator, QoroService
from divi.backends.characterization import CharacterizationOptions, characterize_and_validate
from divi.qprog.algorithms import PCE, QAOA, PolyEncoding, SoftObjective
from divi.qprog.ensembles import BeamSearchStrategy, PartitioningProgramEnsemble
from divi.qprog.optimizers import MonteCarloOptimizer
from divi.qprog.problems import (
    BinaryOptimizationProblem,
    CommunityDecomposer,
    LinearConstraint,
    QUBOPartitioningConfig,
)

print("qoro-divi", version("qoro-divi"))

qoro-divi 0.16.2


In [4]:
USE_CLOUD = False            # False: local Maestro simulator · True: Qoro cloud (uses credits)
RUN_CHARACTERIZATION = True  # Characterization Service calls (cloud; cost scales with QUBO size)
SHOTS = 10_000
MC_POPULATION = 50          # Monte Carlo optimiser: circuits per iteration, sent to the backend as one batch
MC_ITERATIONS = 10          # Monte Carlo optimiser iterations

try:
    service = QoroService()  # reads QORO_API_KEY from .env or the environment
    balance = service.get_credit_balance()
    print("Connected to Qoro. Credit balance:", balance.get("balance", balance))
except (ValueError, requests.RequestException) as err:  # no key, a wrong key, or no connection
    service = None
    print(f"{err}\nNo working API key (notebook 00, section 0.1): cloud cells will be skipped.")
if USE_CLOUD and service is None:
    raise RuntimeError("USE_CLOUD = True needs a working API key: see notebook 00, section 0.1.")


def make_backend(shots: int = SHOTS, force_sampling: bool = False):
    # force_sampling=True: estimate everything from shots instead of exact expectation values
    if USE_CLOUD:
        return QoroService(
            job_config=JobConfig(
                shots=shots, simulator_cluster="qoro_maestro", force_sampling=force_sampling
            )
        )
    return MaestroSimulator(shots=shots, force_sampling=force_sampling)


backend = make_backend()
print("Circuits run on:", type(backend).__name__)

Connected to Qoro. Credit balance: 969.80
Circuits run on: MaestroSimulator


**`solve_partitioned`** is the helper from notebook 00 (section 1.5), with one addition: an optional
`decomposer` argument, for when you want to choose the partitions yourself.

In [5]:
def solve_partitioned(bqm, routine="qaoa", cluster_size=12, n_layers=2, backend=backend, decomposer=None, **routine_kwargs):
    # Partition `bqm`, solve every part with `routine`, and return the ensemble.
    # routine: "qaoa" | "iterative_qaoa" | "pce" | "progressive_pce"
    problem = BinaryOptimizationProblem(
        bqm,
        decomposer=decomposer or CommunityDecomposer(
            QUBOPartitioningConfig(max_n_variables_per_cluster=cluster_size)
        ),
        composer=hybrid.SplatComposer(),
        local_search=True,
    )
    ensemble = PartitioningProgramEnsemble(
        problem=problem,
        quantum_routine=routine,
        n_layers=n_layers,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=backend,
        **routine_kwargs,
    )
    ensemble.run()
    return ensemble

---
## 1 · QOBLIB 08: network design

Given an $n \times n$ traffic demand matrix $t_{ij}$, build a directed network on $n$ nodes in which
**every node has exactly 2 outgoing and 2 incoming links** (no self-loops). Route all traffic, letting
flows split over several paths, and **minimise the largest total flow on any link**. The score is that
maximum × 1000, as an integer. Instances `network05` … `network24` use the top-left $n \times n$
block of one fixed demand matrix.

The problem splits into two parts (credit to the parallel-tempering submission on QOBLIB):

* **Choosing the topology** is combinatorial: one binary variable per possible link, $n(n-1)$ in
  total. This is the QUBO part.
* **Routing on a fixed topology** is a linear program that `qt.route_min_congestion` solves exactly in
  milliseconds. It reproduces every published QOBLIB value.

So the quantum or QUBO solver *proposes* topologies, and the LP *scores* them.

In [6]:
N_NODES = 24
demand = qt.load_network_demand(N_NODES)
print("Demand matrix (traffic from row node to column node):\n", demand)

ref_path = qt.fetch_qoblib_file(f"08-network/solutions/network{N_NODES:02d}.opt.sol")
_, ref_arcs, _ = qt.read_network_solution(ref_path)
print("\nPublished optimal topology:", [(i + 1, j + 1) for i, j in ref_arcs])
print("Its score:", qt.route_min_congestion(demand, ref_arcs).objective)
print("A random valid topology:", qt.route_min_congestion(demand, qt.random_topology(N_NODES, np.random.default_rng(0))).objective)

Demand matrix (traffic from row node to column node):
 [[  0  24  43  23  21  41  61  21  20   0   0   0   0   0  40   0   0   0
    0   0  24  19  23   0]
 [  0   0   0   0  21  18  39  23   0   0  40  19  64  19  17  64  80   0
    0  22  24  18  19   0]
 [ 16   0   0  20  44   0  42  22  20   0   0  23  21   0  40  21  38  97
   22  17  20  37  17  20]
 [ 42  20  18   0  40  57  43  42   0   0  19  17   0  39   0   0  22  41
    0   0  17  42  40  43]
 [  0   0   0  40   0  83  60   0  44   0  37  60   0  40   0   0  40  19
    0  39  41  17   0   0]
 [ 60  18   0  37  18   0  21  41  23  39   0  63  60  39   0  19   0  16
   16   0   0  40   0  16]
 [ 22   0   0   0   0   0   0  61  36  80  96  19  19  41  16   0   0   0
   22   0  43   0  44  22]
 [  0   0  20  19  17  20  40   0   0  60  61   0  20  62  20   0   0  38
    0   0   0   0  24  22]
 [ 21  22  38   0  44  20  40  39   0  36  22  21  19  39  19   0   0  21
   24  16  23  21  37   0]
 [  0  24  23  39  20   0   0  41   

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│    2 demand = qt.load_network_demand(N_NODES)                                                    │
│    3 print("Demand matrix (traffic from row node to column node):\n", demand)                    │
│    4                                                                                             │
│ ❱  5 ref_path = qt.fetch_qoblib_file(f"08-network/solutions/network{N_NODES:02d}.opt.sol")       │
│    6 _, ref_arcs, _ = qt.read_network_solution(ref_path)                                         │
│    7 print("\nPublished optimal topology:", [(i + 1, j + 1) for i, j in ref_arcs])               │
│    8 print("Its score:", qt.route_min_congestion(demand, ref_arcs).objective)                    │
│                                                                                                  │
│ /Users/isaac/Documents/SharQ-Hackathon/qoblib_tools.py:46 in fetch_qoblib_file                   │
│                                                                                                  │
│    43 │   if not dest.exists():                                                                  │
│    44 │   │   dest.parent.mkdir(parents=True, exist_ok=True)                                     │
│    45 │   │   response = requests.get(f"{QOBLIB_RAW}/{rel_path}", timeout=60)                    │
│ ❱  46 │   │   response.raise_for_status()                                                        │
│    47 │   │   tmp = dest.with_name(dest.name + ".part")                                          │
│    48 │   │   tmp.write_bytes(response.content)                                                  │
│    49 │   │   tmp.replace(dest)                                                                  │
│                                                                                                  │
│ /Users/isaac/.cache/uv/archive-v0/2YXCiufWTqK-rxMcFTonQ/lib/python3.12/site-packages/requests/mo │
│ dels.py:1167 in raise_for_status                                                                 │
│                                                                                                  │
│   1164 │   │   │   )                                                                             │
│   1165 │   │                                                                                     │
│   1166 │   │   if http_error_msg:                                                                │
│ ❱ 1167 │   │   │   raise HTTPError(http_error_msg, response=self)                                │
│   1168 │                                                                                         │
│   1169 │   def close(self) -> None:                                                              │
│   1170 │   │   """Releases the connection back to the pool. Once this method has been            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
HTTPError: 404 Client Error: Not Found for url: 
https://raw.githubusercontent.com/ZIB-AOPT/QOBLIB/main/08-network/solutions/network24.opt.sol

For `network05` there are few enough topologies to score every one, giving a ground truth for the
pipeline below.

In [ ]:
rows = [list(itertools.combinations([j for j in range(N_NODES) if j != i], 2)) for i in range(N_NODES)]
all_topologies = []
for choice in itertools.product(*rows):
    arcs = [(i, j) for i, pair in enumerate(choice) for j in pair]
    if qt.is_valid_topology(N_NODES, arcs):
        all_topologies.append(arcs)
all_scores = np.array([qt.route_min_congestion(demand, a).objective for a in all_topologies])
print(f"{len(all_topologies)} valid topologies; best {all_scores.min():.0f}, median {np.median(all_scores):.0f}, worst {all_scores.max():.0f}")

### 1.1 A topology QUBO with Divi's `LinearConstraint`

`qt.build_network_qubo` gives a **surrogate** objective. It rewards links that carry their own demand
directly: $-\sum t_{ij} x_{ij}$, scaled. The real objective needs the LP and is not a QUBO, so
**designing a better surrogate is a big part of the challenge.**

The degree rules are linear equalities. Instead of hand-building penalties, hand them to Divi as
`LinearConstraint`s: `BinaryOptimizationProblem` adds them as squared penalties, and
`problem.is_feasible(bitstring)` checks them. QAOA's `get_top_solutions(feasibility="filter")` keeps
only the feasible samples, ranked by the true objective rather than the penalised one.

In [ ]:
surrogate, degree_penalty, arcs = qt.build_network_qubo(demand)  # arcs[v] = link of variable v
constraints = []
for node in range(N_NODES):
    constraints.append(LinearConstraint({v: 1 for v, (i, j) in enumerate(arcs) if i == node}, "==", 2))  # out
    constraints.append(LinearConstraint({v: 1 for v, (i, j) in enumerate(arcs) if j == node}, "==", 2))  # in
net_problem = BinaryOptimizationProblem(surrogate, constraints=constraints, penalty_weight=2.0)
print(f"{len(arcs)} binary variables → {net_problem.cost_hamiltonian.num_qubits} qubits")

### 1.2 Characterize & validate, then run QAOA

Before spending circuits, send `net_problem` to the **Characterization Service** (notebook 00, section
1.2) with the **published optimal topology as the reference state**. That validates the surrogate: if
the true optimum sits high in the QUBO's energy landscape, no sampler will find it through this QUBO,
and the surrogate needs work. On `network05` the published optimum is the QUBO's ground state.

The constraints count as the penalty component, so the report has a `penalty` section with a
recommended weight, and its `structure` section lists the ten degree rules as *penalty blocks*: groups
of 4 variables that must hold exactly 2 links each. The call can take several minutes. Divi warns that
the QUBO's constant offset isn't sent: the reported energies exclude it, and nothing else changes.

In [ ]:
opt_arcs = set(map(tuple, ref_arcs))  # the published optimal topology, loaded above
ref_bits = "".join("1" if tuple(arc) in opt_arcs else "0" for arc in arcs)

net_report = None
if service is not None and RUN_CHARACTERIZATION:
    net_report = characterize_and_validate(
        net_problem,
        reference_states=[ref_bits],
        service=service,
        options=CharacterizationOptions(ansatz={"layers": 2}),
    )
    display(net_report)
    print("published optimum:  ", net_report.references[0])
    print("penalty section:    ", net_report.penalty)
    print("penalty blocks:     ", len(net_report.structure["penalty_blocks"]))
    print("recommended layers: ", net_report.recommended_layers)
else:
    print("Skipped: no API key, or RUN_CHARACTERIZATION is False.")

QAOA runs twice, as in notebook 00: **blind** (Monte Carlo from random starting angles) and, if you ran the
characterization, **warm-started** from the service's angles at its recommended layer count, plus small random offsets. Every
sample is checked against the degree rules, and every valid topology is scored by the LP.

In [ ]:
def run_network_qaoa(label, n_layers, initial_params=None):
    t0 = time.time()
    qaoa_net = QAOA(
        net_problem,
        n_layers=n_layers,
        optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
        max_iterations=MC_ITERATIONS,
        backend=make_backend(shots=20_000),
        seed=3,
    )
    qaoa_net.run(initial_params=initial_params)
    valid = qaoa_net.get_top_solutions(n=0, feasibility="filter")  # n=0: every feasible sample
    scores = {s.bitstring: qt.route_min_congestion(demand, qt.decode_topology(s.bitstring, arcs)).objective for s in valid}
    p_opt = sum(s.prob for s in valid if scores[s.bitstring] == all_scores.min())
    best = min(scores.values())
    print(f"{label:<24} {qaoa_net.total_circuit_count:>3} circuits, {time.time() - t0:>3.0f}s | {len(valid)} valid topologies, "
          f"{sum(s.prob for s in valid):5.1%} of the probability mass | P(optimal topology) {p_opt:.3%} | "
          f"best sampled: {best:.0f}, rank {(all_scores < best).sum() + 1} of {len(all_topologies)}")


run_network_qaoa("QAOA p=2, blind", n_layers=2)
init = net_report.qaoa_initial_params() if net_report is not None else None
if init is not None:
    p = init.shape[1] // 2
    # One starting point per population member: the service's angles plus small random offsets.
    starts = init + np.random.default_rng(3).normal(scale=0.05, size=(MC_POPULATION, 2 * p))
    starts[0] = init[0]
    run_network_qaoa(f"QAOA p={p}, warm-started", n_layers=p, initial_params=starts)

### 1.3 Scale up: `network12`

132 binary variables. We compare four ways of proposing topologies, each scored by the LP:

* random valid topologies,
* simulated annealing on the surrogate QUBO,
* partitioned QAOA on the surrogate QUBO,
* PCE on the *whole* surrogate QUBO: 132 variables on 16 qubits.

Samples can break the degree rules. Partitions cut straight through them, and single-bit flips
can't fix a node with one link too many when the other end already has exactly two.
`qt.repair_topology` maps any sample to the closest valid topology, keeping as many of its links as
possible, so every sample becomes a scorable candidate.

In [ ]:
N_BIG, BKV_NET = 12, 276474
demand_big = qt.load_network_demand(N_BIG)
sur_big, pen_big, arcs_big = qt.build_network_qubo(demand_big)
net_bqm = sur_big.copy()
pen = pen_big.copy()
pen.scale(1.0)  # penalty weight
net_bqm.update(pen)


def best_topology(samples, arcs, demand):
    # Decode → repair → LP-score every sample; return (best score, best arcs, #distinct topologies).
    n = len(demand)
    topologies = {
        tuple(sorted(qt.repair_topology(n, qt.decode_topology(s, arcs), demand))) for s in samples
    }
    scores = {t: qt.route_min_congestion(demand, t).objective for t in topologies}
    best = min(scores, key=scores.get)
    return scores[best], list(best), len(topologies)


rng = np.random.default_rng(0)
random_best = min(qt.route_min_congestion(demand_big, qt.random_topology(N_BIG, rng)).objective for _ in range(300))
print(f"Random (300 topologies):          {random_best:.0f}")

sa_net = SimulatedAnnealingSampler().sample(net_bqm, num_reads=300, seed=1)
sa_net_score, sa_net_arcs, n_valid = best_topology(sa_net.samples(), arcs_big, demand_big)
print(f"Simulated annealing on surrogate: {sa_net_score:.0f} ({n_valid} distinct topologies)")

t0 = time.time()
net_bqm_n = net_bqm.copy()
net_bqm_n.normalize()
net_ensemble = solve_partitioned(net_bqm_n, "qaoa", cluster_size=12)
net_tops = net_ensemble.get_top_solutions(
    n=100, strategy=BeamSearchStrategy(beam_width=100, n_partition_candidates=10)
)
q_net_score, q_net_arcs, n_valid = best_topology(
    [dict(zip(net_bqm_n.variables, solution)) for solution, _ in net_tops], arcs_big, demand_big
)
print(f"Partitioned QAOA on surrogate:    {q_net_score:.0f} ({n_valid} distinct topologies, {time.time() - t0:.0f}s)")

t0 = time.time()
pce_net = PCE(
    BinaryOptimizationProblem(net_bqm_n),
    encoding=PolyEncoding(),
    objective=SoftObjective(alpha=2.0),
    n_layers=2,
    optimizer=MonteCarloOptimizer(population_size=MC_POPULATION),
    max_iterations=MC_ITERATIONS,
    backend=backend,
    seed=1,
)
pce_net.run()
pce_tops = pce_net.get_top_solutions(n=100, min_prob=0.0, sort_by="energy")
pce_net_score, pce_net_arcs, n_valid = best_topology([s.bitstring for s in pce_tops], arcs_big, demand_big)
print(f"PCE on surrogate ({pce_net.n_qubits} qubits):     {pce_net_score:.0f} ({n_valid} distinct topologies, {time.time() - t0:.0f}s)")
print(f"Best known:                       {BKV_NET}")

### 1.4 Write a submission file

The checker wants **integer** flows, in units of 1/1000. `qt.route_integral` finds integer flows that
achieve the LP score, and `qt.check_network_solution` is a port of QOBLIB's checker.

In [ ]:
best_net_score, best_net_arcs = min(
    (sa_net_score, sa_net_arcs), (q_net_score, q_net_arcs), (pce_net_score, pce_net_arcs), key=lambda x: x[0]
)
routing = qt.route_integral(demand_big, best_net_arcs)
net_path = f"network{N_BIG:02d}.sol"
z = qt.write_network_solution(net_path, N_BIG, routing)
print(f"Wrote {net_path} with objective {z}")
print("Checker:", qt.check_network_solution(net_path, N_BIG, demand_big))

### 1.5 Targets and ideas

Best-known values from the [QOBLIB table](https://github.com/ZIB-AOPT/QOBLIB/blob/main/08-network/solutions/README.md) (check it for updates).

| Instance | Variables | Best known | Status |
|---|---:|---:|---|
| `network05` … `network10` | 20 – 90 | 65 500 … 210 800 | proven optimal: match them |
| `network11`, `network13` … `network23` | 110 – 506 | 238 334 … 659 369 | best known, set by a parallel-tempering submission |
| `network12` | 132 | 276 474 | best known (reference) |
| `network24` | 552 | 663 688 | best known (reference) |

**Ideas to try**
* **Better surrogates.** Reward two-hop paths $x_{ij}x_{jk}$ for large $t_{ik}$. Penalise nodes that
  send a lot of traffic over few links. Learn the weights from LP results.
* **LP feedback (Benders-style).** The LP's dual values tell you which links are congested. Re-weight
  the QUBO and re-solve in a loop.
* **Feasible-by-construction encodings.** A valid topology is two permutations with no fixed points
  that never agree. One-hot permutation encodings with an XY mixer keep QAOA inside the feasible space. The report in section 1.2 suggests the same: an XY mixer over each
  penalty block, started from a state with 2 links.
* **Beat SA first.** On `network12` the plain quantum pipelines above are far from the best known
  value, and simulated annealing on the same surrogate is a strong bar. Closing that gap with better surrogates, penalties, depth or encodings is a good first goal.
* **Quantum proposals inside a local search.** Use QAOA or PCE samples to propose degree-preserving
  2-exchanges, $(a\to b), (c\to d) \Rightarrow (a\to d), (c\to b)$, scored by the LP.
* **Characterize at scale.** Section 1.2 characterizes `network05`. Do the same for the instance you
  are pushing: read `structure` to choose partitions and `penalty` for the weight, and pass your best
  topology as a reference state to check that your surrogate ranks it well.

---
## Handing in

Keep every `.sol` file you write, with the instance, its size and the method and settings that produced
it. **Handing in** at the end of notebook 00 says how to submit.